In [2]:
pip install -U bitsandbytes>=0.46.1
# temp fix

In [3]:
import os
import sys
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from google.colab import drive

# 1. Configurazione del percorso e Cache (Colab o Locale)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    
    # Impostiamo HF_HOME prima di importare hf_hub o transformers
    # In questo modo i modelli vengono salvati su Google Drive e non si riscaricano.
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    print(f"Ambiente Colab e Google Drive configurati. Cache modelli in: {hf_cache_dir}")
except ImportError:
    print("Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.")
     

# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-3.2-1B-Instruct"
print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

label2id = {
    "Clear Reply": 0,
    "Ambivalent Reply": 1,
    "Clear Non-Reply": 2
}
id2label = {v: k for k, v in label2id.items()}


model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,  # Corretto da torch_dtype
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")

Mounted at /content/drive
Ambiente Colab configurato per l'esecuzione.
Acquisizione token da Google Secrets...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Autenticazione Hugging Face completata.
Inizializzazione configurazione BitsAndBytes per meta-llama/Llama-3.2-1B-Instruct...
Trasferimento dei pesi del modello in VRAM...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Caricamento completato con successo. Sistema pronto per l'inferenza.


In [4]:
from datasets import load_from_disk

train_dataset = load_from_disk("dataset/QEvasion/train")
test_dataset = load_from_disk("dataset/QEvasion/test")

print(train_dataset)
print(train_dataset[0])


Dataset({
    features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
    num_rows: 3448
})
{'title': "The President's News Conference in Hanoi, Vietnam", 'date': 'September 10, 2023', 'president': 'Joseph R. Biden', 'url': 'https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-hanoi-vietnam-0', 'question_order': 1, 'interview_question': 'Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?', 'interview_answer': "Well, look, first of all, theI am sincere about getting the relationship right. And o

In [5]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

peft_model = get_peft_model(model, config)

peft_model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,524,480 || trainable%: 0.1377


In [6]:
def tokenize_function(example):

    text = f"Question: {example['question']}\nAnswer: {example['interview_answer']}"

    tokenized_inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=512 # È buona norma impostare un max_length, es. 512 o 1024
    )

    label_str = example['clarity_label']

    # Se per qualche motivo ci fosse un errore nel dato, mettiamo un default o skippiamo
    tokenized_inputs["label"] = label2id.get(label_str, -1)

    return tokenized_inputs

train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

# RIMUOVI GLI ESEMPI NON VALIDI (es. se la label era -1)
train_dataset = train_dataset.filter(lambda x: x["label"] != -1)
test_dataset = test_dataset.filter(lambda x: x["label"] != -1)

# Rimuovi le colonne stringa che non servono più al modello per evitare conflitti nel DataLoader
columns_to_remove = train_dataset.column_names
columns_to_remove.remove("input_ids")
columns_to_remove.remove("attention_mask")
columns_to_remove.remove("label")
train_dataset = train_dataset.remove_columns(columns_to_remove)
test_dataset = test_dataset.remove_columns(columns_to_remove)

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3448 [00:00<?, ? examples/s]

Filter:   0%|          | 0/308 [00:00<?, ? examples/s]

In [7]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3
)

In [11]:
from sklearn.metrics import accuracy_score, f1_score

# Function to compute accuracy
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

In [12]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
50,No log,0.914551,0.794118,0.577931,0.743773
100,No log,0.567871,0.833333,0.706847,0.812567
150,No log,1.153320,0.803922,0.559205,0.739522
200,No log,0.767578,0.843137,0.703488,0.815207
250,No log,0.871094,0.794118,0.521341,0.719724
300,No log,0.415771,0.892157,0.825641,0.884766
350,No log,0.395508,0.892157,0.825641,0.884766
400,No log,0.614746,0.892157,0.825641,0.884766
450,No log,0.376953,0.901961,0.859659,0.901961
500,0.491636,0.600586,0.872549,0.775824,0.856669


TrainOutput(global_step=1056, training_loss=0.32600944782748364, metrics={'train_runtime': 876.0147, 'train_samples_per_second': 4.822, 'train_steps_per_second': 1.205, 'total_flos': 1.2649858728984576e+16, 'train_loss': 0.32600944782748364, 'epoch': 3.0})

In [14]:
results = trainer.evaluate()

print("Accuracy:", results["eval_accuracy"])

Accuracy: 0.9019607843137255


In [15]:
# Scegli il nome della cartella in cui salvare il modello (verrà creata se non esiste)
# Visto che sei già posizionato nel tuo Google Drive, verrà salvata lì e non la perderai!
save_directory = "Llama-3.2-1B-Instruct-Clarity"

# Salva i pesi del modello addestrato
trainer.save_model(save_directory)

# Salva anche il tokenizer (fondamentale per riutilizzare il modello dopo)
tokenizer.save_pretrained(save_directory)

print(f"Modello e tokenizer salvati con successo in: {save_directory}")

Modello e tokenizer salvati con successo in: Llama-3.2-1B-Instruct-Clarity
